# ALICE Run 3 simulation hands-on — 0. Setup

This notebook checks that your SWAN session can run the ALICE simulation software from CVMFS.

**Session settings** (in the SWAN start page, before starting the session):

| Setting | Value |
|---|---|
| Software stack | latest default LCG release (the notebooks do **not** use it for O2) |
| Platform | AlmaLinux 9 |
| Number of cores | **4** |
| Memory | **16 GB** |
| Environment script | *(leave empty)* |

**How O2 is used here.** The SWAN Python kernel uses the LCG software stack, which has its own ROOT and Python.
O2 brings different versions. To avoid conflicts, every O2 command runs through the small wrapper **`o2exec`** in this project.
It runs the command in a clean environment loaded with `alienv` from `/cvmfs/alice.cern.ch`:

```bash
o2exec o2-sim -n 5 -g pythia8pp        # = alienv setenv VO_ALICE@O2sim::<tag> -c o2-sim ...
```

Cells starting with `%%bash` are shell cells. Each one runs in its own shell.

In [ ]:
from swan_env import *

## 1. Resources of this session

In [ ]:
%%bash
echo "cores : $(nproc)"
free -g | head -2
echo
df -h /dev/shm /tmp | sed 's/^/  /'
echo
ls /cvmfs/alice.cern.ch/bin/alienv && echo "CVMFS alice.cern.ch: OK"

What to expect:
- **4 cores and ~16 GB** of memory. The O2DPG exercises need this.
- **`/tmp`** with at least ~20 GB free. Simulations run there (`SIMTUT_DIR`), not on CERNBox.
- **`/dev/shm`**: the DPL workflows use shared memory. `o2exec` sets the segment size (`SHMSIZE`) to 3/4 of `/dev/shm` automatically.

## 2. Available O2sim releases

The tutorial uses the tag in `swan_env.py` (`O2SIM_TAG`). To change it for all notebooks, edit that file.

In [ ]:
%%bash
/cvmfs/alice.cern.ch/bin/alienv q 2>/dev/null | grep -E '^VO_ALICE@O2sim::v2026' | tail -5

## 3. Load O2 for the first time

The first call downloads the software into the local CVMFS cache, so it can take **a few minutes**. Later calls are fast.

In [ ]:
%%bash
time o2exec bash -c 'echo "O2_ROOT    = $O2_ROOT"; echo "O2DPG_ROOT = $O2DPG_ROOT"; which o2-sim; root-config --version'

In [ ]:
%%bash
o2exec o2-sim --help | head -40

## 4. AliEn token (needed for all exercises)

When it transports particles, `o2-sim` loads the detector **alignment** from CCDB, and the O2DPG workflows read many calibration objects. Many of these objects are stored on the GRID, which needs a valid **AliEn token**.
Creating the token asks for your grid-certificate password, so it is done in a **SWAN terminal**:
*New → Terminal* in the SWAN file browser.

Your certificate must be in `~/.globus/` on CERNBox (`usercert.pem`, `userkey.pem`).
Paste this into the terminal, replacing `<project>` with the path printed by the first cell of this notebook:

```bash
# private key permissions cannot be enforced on CERNBox: use a local copy
mkdir -p /tmp/$USER/globus && cp ~/.globus/user{cert,key}.pem /tmp/$USER/globus/
chmod 400 /tmp/$USER/globus/userkey.pem
export X509_USER_CERT=/tmp/$USER/globus/usercert.pem X509_USER_KEY=/tmp/$USER/globus/userkey.pem

<project>/o2exec alien-token-init $USER
```

The token is written to `/tmp` and is valid for the whole SWAN session. Check it here:

In [ ]:
%%bash
o2exec alien-token-info

✅ If the token is valid, continue with **`01_o2sim_basics.ipynb`**.

> If your SWAN session is restarted, `/tmp` is wiped: create the token again. Simulation outputs in `/tmp` are also lost.
> Each notebook copies its important results to `~/simtutorial_results` on CERNBox.